In [1]:
import numpy as np
from itertools import combinations
import subprocess

In [2]:
## read output files generated from the command line output from Sedonu simulations.
## extract total escaped energy, number emitted per flavor, number escaped per flavor, total end energy, time 
## find the averages and the standard deviation (of each variable) for a set of simulations with the exact same parameters.
## Plot these averages with error bars as a function of increasing tau, then as a function of packet weight.

In [3]:
# significance testing naming scheme 
name_base = 'tau5_pw100_'
sample_size = 5
filenames = [name_base+str(i+1)+'.txt' for i in (range(0,sample_size))] 
dirname = '../0km/luminosity_check/tau_withGR/tau5/'

In [4]:
print(filenames)

['tau5_pw100_1.txt', 'tau5_pw100_2.txt', 'tau5_pw100_3.txt', 'tau5_pw100_4.txt', 'tau5_pw100_5.txt']


In [5]:
data = {
    "l_end": [],
    "l_esc": [],
    "l_roul": [],
    "nve_emitted": [],
    "nva_emitted": [],
    "nvx_emitted": [],
    "sim_times": [],
}

In [6]:
# significance testing 
for filename in filenames:
    with open (dirname + filename, "r") as file:
        lines = file.readlines()
    
    for line in lines: 
        if 'TOTAL PARTICLE END' in line:
            data["l_end"].append(line.strip('#').strip().split(' ')[0])
        elif 'TOTAL ROULETTED' in line: 
            data["l_roul"].append(line.strip('#     --> ').split(' ')[0])
        elif 'TOTAL ESCAPED' in line: 
            data["l_esc"].append(line.strip('#     --> ').split(' ')[0])
        elif 'N_emit (lab)' in line:
            data["nve_emitted"].append(line.strip('#   {  ').split()[0])
            data["nva_emitted"].append(line.strip('#   {  ').split()[1])
            data["nvx_emitted"].append(line.strip('#   {  ').split()[2].split()[0])
        elif 'CALCULATION took' in line:
            data["sim_times"].append(line.split(' ')[3])
    file.close()
    
data = {k: np.array([float(x) for x in v]) for k,v in data.items()} 

In [7]:
# using the standard error of the mean, instead of the standard deviation of the set 
for key, value in data.items():
    print(f'Average, stdev for {key} for {sample_size} simulations: {np.average(value)} , {np.std(value, ddof=1) / np.sqrt(sample_size)}')
    if "l_esc" in key:
        l_esc = list(value)
print("\nRecorded luminositites:\n")

for n in np.arange(0, sample_size):
    print(f"L_esc #{n+1}: {l_esc[n]} erg/s")


Average, stdev for l_end for 5 simulations: 1.8249523999999998e+60 , 1.0777046381758129e+60
Average, stdev for l_esc for 5 simulations: 1.315472e+53 , 6.672856845106899e+52
Average, stdev for l_roul for 5 simulations: 1.8249522e+60 , 1.0777047129408126e+60
Average, stdev for nve_emitted for 5 simulations: 1.22481486e+64 , 5.499487866038069e+63
Average, stdev for nva_emitted for 5 simulations: 1.1200774e+61 , 1.5297262906533308e+60
Average, stdev for nvx_emitted for 5 simulations: 1.457244e+61 , 5.754842626866527e+59
Average, stdev for sim_times for 5 simulations: 252.06 , 3.3837257572090556

Recorded luminositites:

L_esc #1: 1.35099e+53 erg/s
L_esc #2: 1.83728e+52 erg/s
L_esc #3: 9.26995e+52 erg/s
L_esc #4: 2.73137e+52 erg/s
L_esc #5: 3.84251e+53 erg/s


In [8]:
# Method 1: T-test for samples with the same run configuration
# Examine the pair-wise differences of all measurements, compare to the ratio of the standard deviation of the differences
# the "difference" distribution is a gaussian centered on 0 

# calculate all-pairwise differences between luminosity measurements
differences = [abs(a-b) for a,b in combinations(l_esc, 2)]
# find the standard deviation for all measurements
sigma_del = np.std(differences, ddof=1)
# correct the z-score by sqrt(2) given that we are comparing two measurements (not N_total?)
# for z score > 2 or 3 sigma, we have a problem?
z_scores = [np.sqrt(2)* d/sigma_del for d in differences]
print(z_scores)

[1.2253520424537314, 0.4450955648690439, 1.1314935078970116, 2.615513158840364, 0.7802564775846874, 0.0938585345567196, 3.8408652012940956, 0.6863979430279679, 3.0606087237094073, 3.747006666737376]


In [9]:
# statistical signficance of Tau=1 and Tau=5

delta_mu = abs(1.315472e+53 - 8.996292e+52)
std_null = np.sqrt((8.374060806048639e+51)**2 + (6.672856845106899e+52)**2)
print(f'Difference in means: {delta_mu}\nNull hypothesis standard deviation: {std_null}\nDelta/sigma: {delta_mu/std_null}')

Difference in means: 4.1584280000000003e+52
Null hypothesis standard deviation: 6.725196459518785e+52
Delta/sigma: 0.6183355423192429


In [41]:
# linear interpolation of the luminosities in order to analyze convergence

# luminosity convergence naming scheme
base_dir = '/mnt/scratch/MRSN_crossing/SedonuGR/BH2_50ms_coreemit'
command = f"ls {base_dir}"
result = subprocess.check_output(command, shell=True, text=True)
radii = []
filenames = []
for content in [k for k in result.split('\n') if 'km' in k]:
    radii.append(content)
radii.remove("0km")
filenames =[f'{base_dir}/{r}/output.txt' for r in radii]

In [45]:
l_data = {
    "l_end": [],
    "l_esc": [],
    "l_roul": [],
    "radius": [],
    "sim_times": [],
}

In [47]:
# luminosity convergence testing 

# significance testing 
for filename in filenames:
    print(filename)
    with open (filename, "r") as file:
        lines = file.readlines()
    for line in lines: 
        if 'TOTAL PARTICLE END' in line:
            l_data["l_end"].append(line.strip('#').strip().split(' ')[0])
        elif 'TOTAL ROULETTED' in line: 
            l_data["l_roul"].append(line.strip('#     --> ').split(' ')[0])
        elif 'TOTAL ESCAPED' in line: 
            l_data["l_esc"].append(line.strip('#     --> ').split(' ')[0])
        elif 'CALCULATION took' in line:
            l_data["sim_times"].append(line.split(' ')[3])
    file.close()
    
    with open(f'{base_dir}/{radii[filenames.index(filename)]}/param.lua', 'r') as param_file:
        lines = param_file.readlines()
        parameters=[i for i in lines if '-- ' not in i]
        parameters=[i for i in parameters if i.startswith(" ") == False]
        for p in parameters:
            if 'r_core' in p:
                k = p.split(" = ")[0]
                v = p.split(" = ")[-1]
            print(k, v)
            l_data['radius'].append(v)
    file.close()

    
l_data = {k: np.array([float(x) for x in v]) for k,v in data.items()} 

/mnt/scratch/MRSN_crossing/SedonuGR/BH2_50ms_coreemit/17km/output.txt
r_core 35.445649716860956e5



AttributeError: 'str' object has no attribute 'append'